# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mashfiqmahi/assignment_FLyRank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
%pip -q install duckdb huggingface_hub pandas

import os, duckdb, pandas as pd, numpy as np
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

df = con.sql("""
WITH monthly AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions,
        SUM(gsc_clicks)      FILTER (WHERE gsc_data_available IS TRUE) AS clicks,
        SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS sum_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
)
SELECT
    m.content_hash_id,
    m.impressions,
    CASE WHEN m.impressions > 0 THEN m.clicks * 100.0 / m.impressions ELSE NULL END AS ctr_pct,
    CASE WHEN m.impressions > 0 THEN m.sum_position * 1.0 / m.impressions ELSE NULL END AS avg_position,
    DATE_DIFF('day', d.content_created_date, DATE '2026-03-31') AS days_since_created,
    d.content_type
FROM monthly m
JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') d
    ON m.content_hash_id = d.content_hash_id
""").df()

df["days_since_created"] = df["days_since_created"].clip(lower=0)
print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 6)


,content_hash_id,impressions,ctr_pct,avg_position,days_since_created,content_type
0,content_b7e512995f79d5a6,1140.0,0.175439,4.450877,396,keyword article
1,content_05597932fe4da067,57.0,0.000000,2.298246,396,keyword article
2,content_905aa32a0230694e,149.0,0.000000,5.637584,396,keyword article
3,content_05434271b257bb68,1421.0,0.422238,6.906404,396,keyword article
4,content_d056587ff7faca0c,2770.0,0.577617,3.950542,396,keyword article


In [15]:
visible = df[df["impressions"] > 0].copy()

# Bucket pages into age tiers
visible["age_tier"] = pd.cut(
    visible["days_since_created"],
    bins=[-1, 90, 180, 365, 10000],
    labels=["0-90 days", "91-180 days", "181-365 days", "365+ days"]
)

signal1_table = visible.groupby("age_tier", observed=True).agg(
    n=("impressions", "size"),
    median_impressions=("impressions", "median"),
    median_ctr_pct=("ctr_pct", "median")
).round(2)

print(signal1_table)

                  n  median_impressions  median_ctr_pct
age_tier                                               
0-90 days     57735               175.0             0.0
91-180 days   26247               239.0             0.0
181-365 days  71046               128.0             0.0
365+ days     21710               244.0             0.0


In [16]:
pos_valid = df[(df["avg_position"] > 0) & (df["impressions"] > 0)].copy()

pos_valid["position_tier"] = pd.cut(
    pos_valid["avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["top_3", "page_1 (4-10)", "page_2 (11-20)", "deep (20+)"]
)

signal2_table = pos_valid.groupby("position_tier", observed=True).agg(
    n=("ctr_pct", "size"),
    median_ctr_pct=("ctr_pct", "median")
).round(3)

print(signal2_table)

                    n  median_ctr_pct
position_tier                        
top_3           17426             0.0
page_1 (4-10)   83288             0.0
page_2 (11-20)  29922             0.0
deep (20+)      44668             0.0


In [17]:
# First, just confirm the zero-inflation theory
print("Overall CTR stats:")
print(visible["ctr_pct"].describe())
print("\n% of visible pages with CTR exactly 0:", (visible["ctr_pct"] == 0).mean().round(3))

Overall CTR stats:
count    176738.000000
mean          0.459397
std           3.775992
min           0.000000
25%           0.000000
50%           0.000000
75%           0.215796
max         100.000000
Name: ctr_pct, dtype: float64

% of visible pages with CTR exactly 0: 0.611


In [18]:
visible.head()

,content_hash_id,impressions,ctr_pct,avg_position,days_since_created,content_type,age_tier
0,content_b7e512995f79d5a6,1140.0,0.175439,4.450877,396,keyword article,365+ days
1,content_05597932fe4da067,57.0,0.000000,2.298246,396,keyword article,365+ days
2,content_905aa32a0230694e,149.0,0.000000,5.637584,396,keyword article,365+ days
3,content_05434271b257bb68,1421.0,0.422238,6.906404,396,keyword article,365+ days
4,content_d056587ff7faca0c,2770.0,0.577617,3.950542,396,keyword article,365+ days


In [19]:
# First, just confirm the zero-inflation theory
print("Overall CTR stats:")
print(visible["ctr_pct"].describe())
print("\n% of visible pages with CTR exactly 0:", (visible["ctr_pct"] == 0).mean().round(3))

Overall CTR stats:
count    176738.000000
mean          0.459397
std           3.775992
min           0.000000
25%           0.000000
50%           0.000000
75%           0.215796
max         100.000000
Name: ctr_pct, dtype: float64

% of visible pages with CTR exactly 0: 0.611


In [20]:
def bucket_stats(g):
    total_clicks = (g["ctr_pct"] * g["impressions"] / 100).sum()  # reconstruct clicks
    total_impr = g["impressions"].sum()
    weighted_ctr = total_clicks / total_impr * 100 if total_impr > 0 else np.nan
    return pd.Series({
        "n": len(g),
        "median_impressions": g["impressions"].median(),
        "median_ctr_pct": g["ctr_pct"].median(),
        "mean_ctr_pct": g["ctr_pct"].mean(),
        "pct_zero_ctr": (g["ctr_pct"] == 0).mean(),
        "weighted_ctr_pct": weighted_ctr,
    })

signal1_table = visible.groupby("age_tier", observed=True).apply(bucket_stats).round(3)
print("Signal 1 — staleness vs performance:")
print(signal1_table)

signal2_table = pos_valid.groupby("position_tier", observed=True).apply(bucket_stats).round(3)
print("\nSignal 2 — position vs CTR:")
print(signal2_table)

Signal 1 — staleness vs performance:
                    n  median_impressions  median_ctr_pct  mean_ctr_pct  \
age_tier                                                                  
0-90 days     57735.0               175.0             0.0         0.384   
91-180 days   26247.0               239.0             0.0         0.319   
181-365 days  71046.0               128.0             0.0         0.621   
365+ days     21710.0               244.0             0.0         0.301   

              pct_zero_ctr  weighted_ctr_pct  
age_tier                                      
0-90 days            0.587             0.325  
91-180 days          0.590             0.254  
181-365 days         0.637             0.292  
365+ days            0.609             0.286  

Signal 2 — position vs CTR:
                      n  median_impressions  median_ctr_pct  mean_ctr_pct  \
position_tier                                                               
top_3           17426.0               316.5    

/tmp/ipykernel_2503/2803818392.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  signal1_table = visible.groupby("age_tier", observed=True).apply(bucket_stats).round(3)
/tmp/ipykernel_2503/2803818392.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  signal2_table = pos_valid.groupby("position_tier", observed=True).apply(bucket_stats).round(3)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1 — Staleness vs. performance (flag-linked: behind FlyRank's refresh flags)
Claim: "older pages perform worse."
Test: pages bucketed by days_since_created, weighted CTR ( total clicks / total
impressions) per bucket, since raw CTR is heavily zero-inflated (61% of visible
pages have exactly 0 clicks this month — median CTR is 0.0 in every bucket and
uninformative on its own).
Result: weighted CTR = 0.325 / 0.254 / 0.292 / 0.286 across increasing age tiers
(n = 57,735 / 26,247 / 71,046 / 21,710) — no clean decline, values bounce non-
monotonically. Median impressions also bounce (175 / 239 / 128 / 244).
Verdict: MIXED (leaning FALSE). Staleness alone is not a reliable performance
signal in this slice — the baseline rule below does NOT weight it heavily.

Signal 2 — Position vs. CTR (flag-linked: behind FlyRank's CTR-fix logic)
Claim: "pages ranking worse get fewer clicks per impression."
Test: pages bucketed by avg_position, weighted CTR per bucket.
Result: weighted CTR = 0.388 / 0.325 / 0.316 / 0.136 across top_3 / page_1 /
page_2 / deep tiers (n = 17,426 / 83,288 / 29,922 / 44,668) — steady decline,
confirmed further by pct_zero_ctr rising 52% -> 75% for the deepest tier.
Verdict: CONFIRMED. This is the real, load-bearing signal for the rule below.

My rule, in plain words: A page is worth an editor's review if it still gets
meaningful search traffic (visible), AND its click-through rate is worse than
what other pages at its OWN position tier typically achieve — i.e. it is
underperforming its ranking, not just old. Staleness is checked but deliberately
NOT used as a rule input, since Signal 1 didn't support it.

Reason code: "weak_ctr_for_position" — the page gets a below-benchmark CTR
for its position tier, meaning the ranking is fine but something else (title,
snippet, relevance) is likely costing clicks.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [22]:
df["position_tier"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["top_3", "page_1 (4-10)", "page_2 (11-20)", "deep (20+)"]
)

# Benchmark CTR per tier, taken directly from Signal 2 weighted results
benchmark_ctr = {
    "top_3": 0.388,
    "page_1 (4-10)": 0.325,
    "page_2 (11-20)": 0.316,
    "deep (20+)": 0.136,
}
df["benchmark_ctr_pct"] = df["position_tier"].map(benchmark_ctr).astype(float)

# The rule, in code
df["visible_flag"] = (df["impressions"] >= 200).astype(int)
df["weak_ctr_flag"] = (df["ctr_pct"] < df["benchmark_ctr_pct"]).astype(int)

df["score"] = df["visible_flag"] * df["weak_ctr_flag"] * df["impressions"].fillna(0)

df["reason_code"] = np.where(
    (df["visible_flag"] == 1) & (df["weak_ctr_flag"] == 1),
    "weak_ctr_for_position",
    "no_action"
)
df["action"] = np.where(df["score"] > 0, "review", "monitor")

import os
os.makedirs("work/outputs", exist_ok=True)
df_sorted = df.sort_values("score", ascending=False)
df_sorted.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Queue written: {len(df_sorted):,} rows")
print(f"Pages flagged for review: {(df_sorted['action']=='review').sum():,}")
df_sorted[["content_hash_id","position_tier","impressions","ctr_pct",
           "benchmark_ctr_pct","score","reason_code","action"]].head(10)

Queue written: 331,437 rows
Pages flagged for review: 57,496


,content_hash_id,position_tier,impressions,ctr_pct,benchmark_ctr_pct,score,reason_code,action
58741,content_e8a52cf3d5988c07,page_2 (11-20),244931.0,0.273138,0.316,244931.0,weak_ctr_for_position,review
211693,content_0e03de7680314cd5,top_3,221310.0,0.325336,0.388,221310.0,weak_ctr_for_position,review
45375,content_44f34c0a90047651,top_3,212404.0,0.011299,0.388,212404.0,weak_ctr_for_position,review
211658,content_8d7d99f109e19aa2,top_3,203497.0,0.142017,0.388,203497.0,weak_ctr_for_position,review
207072,content_36e53e9c707674fc,deep (20+),194579.0,0.124371,0.136,194579.0,weak_ctr_for_position,review
59641,content_b99ea6861864dea5,page_1 (4-10),194337.0,0.185760,0.325,194337.0,weak_ctr_for_position,review
211688,content_4ffe18112a5642e3,top_3,186983.0,0.313397,0.388,186983.0,weak_ctr_for_position,review
225011,content_acbcc847f8996314,page_1 (4-10),170808.0,0.153389,0.325,170808.0,weak_ctr_for_position,review
281245,content_471d9cabce329a66,page_1 (4-10),164885.0,0.240167,0.325,164885.0,weak_ctr_for_position,review
671,content_fd2117c2c6790e4b,page_1 (4-10),151166.0,0.269902,0.325,151166.0,weak_ctr_for_position,review


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-10 review (each: action, why flagged, what would make it wrong)

1. content_e8a52cf3d5988c07 — review. 244,931 impressions, CTR 0.273 vs page_2
   benchmark 0.316 (~14% short). Would be wrong if: this small a gap is just normal
   week-to-week variance, not a real snippet/title problem — weakest-margin pick
   in the top 10 despite ranking #1 by score.

2. content_0e03de7680314cd5 — review. Top_3 position, CTR 0.325 vs benchmark 0.388
   (~16% short). Would be wrong if: top_3 CTR benchmarks vary a lot by query type
   (e.g. informational vs. navigational), and this page's true peer group has a
   lower natural ceiling than the tier average.

3. content_44f34c0a90047651 — review. Top_3 position, CTR only 0.011 vs benchmark
   0.388 (~97% short) — the strongest, most credible pick in this list. Would be
   wrong if: this near-zero CTR is a tracking/measurement glitch rather than real
   user behavior — worth a manual GSC check before assuming it's a content problem.

4. content_8d7d99f109e19aa2 — review. Top_3, CTR 0.142 vs 0.388 (~63% short).
   Would be wrong if: shares a template/domain issue with #3 — if several top_3
   pages from the same client all show this pattern, it's a systemic tracking
   issue, not independent content problems.

5. content_36e53e9c707674fc — review. Deep(20+) position, CTR 0.124 vs benchmark
   0.136 — only ~9% short, the smallest gap in the top 10, ranked #5 purely
   because of very high impressions (194,579). Would be wrong if: this is normal
   noise around an already-low benchmark — this pick exposes a real weakness in
   the scoring formula: it rewards volume regardless of how small the shortfall is.

6. content_b99ea6861864dea5 — review. Page_1, CTR 0.186 vs benchmark 0.325
   (~43% short). Would be wrong if: manual check shows the title/meta look fine —
   might mean the topic itself has low intent-to-click regardless of position.

7. content_4ffe18112a5642e3 — review. Top_3, CTR 0.313 vs 0.388 (~19% short).
   Would be wrong if: this is within normal noise for a top_3 page — moderate
   gap, not as convincing as #3 or #4.

8. content_acbcc847f8996314 — review. Page_1, CTR 0.153 vs 0.325 (~53% short).
   Would be wrong if: this page recently moved INTO page_1 and simply hasn't
   accumulated clicks yet — a new entrant isn't the same as a broken page.

9. content_471d9cabce329a66 — review. Page_1, CTR 0.240 vs 0.325 (~26% short).
   Would be wrong if: the query intent behind this page is naturally low-CTR
   (e.g. purely informational), not something a snippet rewrite would fix.

10. content_fd2117c2c6790e4b — review. Page_1, CTR 0.270 vs 0.325 (~17% short).
    Would be wrong if: small gap, likely normal variance rather than a real,
    fixable underperformance — similar caveat to #1.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: the scoring formula (visible_flag * weak_ctr_flag * impressions) ranks
purely by impression volume once a page clears the "below benchmark" threshold —
it does not weight HOW FAR below benchmark a page is. This produced weak picks like
#1, #5, #7, #10 above, where the CTR shortfall was small (9-19%) but huge impression
counts still pushed them into the top 10, ahead of more severe underperformers.
A future improvement: multiply the score by the relative gap
(benchmark_ctr_pct - ctr_pct) / benchmark_ctr_pct, not just impressions alone —
left as an honest limitation of this week's deliberately-simple baseline, not fixed
here, since next week's model is explicitly meant to improve on exactly this kind
of blunt-instrument scoring.

Leakage check: confirmed none of trend_direction, trend_pct, is_declining_label,
is_published, is_deleted were used as rule inputs. Rule uses only impressions,
ctr_pct, avg_position (all observed March 2026 performance) plus the position_tier
derived from avg_position and the benchmark_ctr_pct computed from Signal 2's audit —
no future-window or label-derived information anywhere in the score.

In [24]:
rule_inputs = ["impressions", "ctr_pct", "avg_position", "position_tier", "benchmark_ctr_pct"]
forbidden = ["trend_direction", "trend_pct", "is_declining_label", "is_published", "is_deleted"]

print("Rule inputs used:", rule_inputs)
for col in forbidden:
    print(f"{col} used in rule? {col in rule_inputs}")

Rule inputs used: ['impressions', 'ctr_pct', 'avg_position', 'position_tier', 'benchmark_ctr_pct']
trend_direction used in rule? False
trend_pct used in rule? False
is_declining_label used in rule? False
is_published used in rule? False
is_deleted used in rule? False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.